# Basic fitting in **EasyDynamics**
We here show how to use **EasyDynamics** to fit a simple synthetic data set. We start with the simplest possible example; later tutorials will be more involved.

The general procedure is to create an `Experiment` object to hold the data, a `SampleModel` to describe the model, and `Analysis` to fit the model to the data.

In [1]:
# Imports
import pooch
import scipp as sc

import easydynamics as edyn
import easydynamics.sample_model as sm
from easydynamics.analysis.analysis import Analysis

# Make the plots interactive
%matplotlib widget

We first create an `Experiment` object to contain the data. The data must either be a `hdf5` file or a `scipp.DataArray`; in both cases it must have coordinates `Q` and `energy`. We here use Pooch to download an example data set.

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    If you have one or more data files that you want to fit using EasyDynamics, you can reach out to us e.g. at henrik.jacobsen@ess.eu for help.
  </div>
</details>

We give the `Experiment` a `display_name` which is used as the title when plotting the data.

In [2]:
# Load the data
experiment = edyn.Experiment(display_name='Tutorial')

file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/tutorial0/docs/docs/tutorials/data/fake_simple_data.hdf5',
    known_hash='b49944c4447e69be4d30d1bed935173c4a1727c25a347285cbb156edc76ee261',
)

experiment.load_hdf5(filename=file_path)

We can visualize the data in multiple ways, relying on plopp: https://scipp.github.io/plopp/

For now, let us plot the data using a slicer showing intensity as function of `energy` for various `Q`. You can dragg the slider to choose which $Q$ is displayed. Notice that the title is the `display_name` that we gave the `Experiment`.

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    You can change the title by running `experiment.display_name="new title"`
  </div>
</details>

In [3]:
experiment.plot_data(slicer=True)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The data looks somewhat noisy. To improve statistics, we can rebin it. The method takes a dictionary of coordinates and the new bins. The new bins can be given as an integer number of bins or as a `scipp.Variable` (https://scipp.github.io/generated/classes/scipp.Variable.html#scipp.Variable) of bin-edges. For now, we just use integers, choosing 16 bins in $Q$ and 256 bins in energy.

In [4]:
experiment.rebin({'Q': 16, 'energy': 256})
experiment.plot_data(slicer=True)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    You may have noted that the y axis is automatically scaled to include all data for each Q. If you do not want this, use `autoscale=False` in the plot command.
  </div>
</details>

In this data we see a single Gaussian shaped peak and a background that seems to be zero on average. We now want to fit this data, e.g. to determine how the Gaussian changes with $Q$. We define a `Gaussian` like this:

In [5]:
gaussian = sm.Gaussian(display_name='Gaussian', area=1, width=0.05)

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    You can access and change the values of the parameters of the Gaussian like this:
    `gaussian.area=0.5`
  </div>
</details>

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    The following functions are implemented: Gaussian, Lorentzian, Voigt, Delta Function, Damped Harmonic Oscillator, Polynomial. If you need a different model, you can define your own using ExpressionComponent
  </div>
</details>

We would like to fit this model to our data. However, it would be tedious to copy/paste the model for 16 values of $Q$, especially when the model grows more complicated. Instead, **EasyDynamics** handles this for us; we simply have to create a `SampleModel` and pass it our component.

In [6]:
model = sm.SampleModel(components=gaussian)

Our `SampleModel` does not yet have any $Q$ values. We could tell it what $Q$ is by using the `Q` attribute.
```python
Q_values=sc.linspace(start=0.2,stop=2.1,num=16,unit='1/angstrom', dim='Q')

model.Q=Q_values
```

This would generate a copy of the model for each of the values of $Q$. 

However, since we want the $Q$ values to match those of the experiment, it is much easier to let **EasyDynamics** handle it. To do this, we make an `Analysis` object and give it out experiment and sample model:

In [7]:
analysis = Analysis(
    experiment=experiment,
    sample_model=model,
)

A lot happens under the hood in this step. It automatically extracts the $Q$ values from the experiment and passes them to our `SampleModel`, which in turn generates a copy of the `Gaussian` for each $Q$. We furthermore create an `Analysis1d` object for each $Q$. You will usually not need to think too hard about any of this, since you will mostly interact with your data and model through the `Analysis` object, but it can be good to know what is going on.

We can access a list of the the `Analysis1d` objects like this:


In [8]:
analysis_list = analysis.analysis_list

Let us plot our data and model using the plopp slicer:


In [9]:
analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The model is of course not yet particularly good, since we didn't try very hard to guess what the parameters should be. However, we can now start fitting it using the `fit` method.

Let us first fit a single $Q$ index and plot the data and model to see how it looks. We choose an arbitrary $Q$ and plot only that one

In [10]:
fit_result_independent_single_Q = analysis.fit(Q_index=5)
analysis.plot_data_and_model(Q_index=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The fit looks very good. We can get a list of the parameters for this fit by accesing the corresponding `Analysis1d` object:

In [11]:
analysis.analysis_list[5].get_all_parameters()

[<Parameter 'Gaussian area': 3.5814 ± 0.0570 meV, bounds=[0.0:inf]>,
 <Parameter 'Gaussian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Gaussian width': 0.0985 ± 0.0018 meV, bounds=[1e-10:inf]>,
 <Parameter 'energy_offset': -0.0002 ± 0.0017 meV, bounds=[-inf:inf]>]

<details>
  <summary><strong>Note</strong></summary>
<div style="border-left: 4px solid #2196F3; background:#e3f2fd; padding:10px;">
You may note that the center of the Gaussian is fixed, and that we fitted an `energy_offset`. We discuss this in the following tutorial.
</div>
</details>

Since the fit looked good, we can now fit all $Q$. We also plot the result, again using the slicer.

In [12]:
fit_result_all_Q = analysis.fit()
analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

Information about the fit is stored in the output. It is stored as a list of **EasyScience** `FitResult`s. Here we show a few of the relevant properties:

In [13]:
print(f'The reduced chi-squared value for Q_index=5 is: {fit_result_all_Q[5].reduced_chi2}')

print(f'The minimizer engine is: {fit_result_all_Q[5].minimizer_engine}')

The reduced chi-squared value for Q_index=5 is: 0.9807375513271296
The minimizer engine is: <class 'easyscience.fitting.minimizers.minimizer_lmfit.LMFit'>


It can be nice to inspect the fit parameters to look for trends, and sometimes continue working with them. To do this, we convert the parameters to a scipp Dataset (https://scipp.github.io/generated/classes/scipp.Dataset.html#scipp.Dataset)

In [14]:
analysis.parameters_to_dataset()

<scipp.Dataset>
Dimensions: Sizes[Q:16, ]
Coordinates:
* Q                         float64           [1/Å]  (Q)  [0.259375, 0.378125, ..., 1.92188, 2.04063]
Data:
  Gaussian area             float64            [meV]  (Q)  [3.66804, 3.72543, ..., 3.41125, 3.26164]  [0.00395623, 0.00323745, ..., 0.00421608, 0.00418692]
  Gaussian center           float64            [meV]  (Q)  [0, 0, ..., 0, 0]  [0, 0, ..., 0, 0]
  Gaussian width            float64            [meV]  (Q)  [0.0992222, 0.0982519, ..., 0.100922, 0.0993596]  [3.86336e-06, 2.37053e-06, ..., 4.88825e-06, 5.21159e-06]
  energy_offset             float64            [meV]  (Q)  [-0.000333302, -0.000418205, ..., -0.0035326, 7.40604e-05]  [3.83889e-06, 3.63992e-06, ..., 4.74489e-06, 5.08538e-06]

We can also plot the parameters as a function of `Q` using the `plot_parameters` method.

In [15]:
analysis.plot_parameters(names=['Gaussian area'])

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

If you wish to customise the plot beyond what is immediately possible with EasyDynamics, you can get the data and model as a scipp datagroup. You may evaluate the model at different energies than the data like this.

In [16]:
energy = sc.linspace('energy', -3.5, 3.5, num=1001, unit='meV')
data_and_model = analysis.data_and_model_to_datagroup(energy=energy)

It will soon be possible to use **EasyDynamics** to fit these parameters to e.g. a polynomial.